# Clasificación v2 de movilidad con vLLM (Kaggle, GPU T4 x2)

Corre `clasificar_vllm.py`: una copia de Qwen2.5-7B-Instruct AWQ por GPU, salida JSON guiada y cascada en dos etapas (relevancia en todo, clasificación solo en las relevantes). Las reglas salen de `etiquetado/guia-movilidad.md`.

**Antes de correr:** Accelerator `GPU T4 x2`, Internet On, y como input el Kaggle Dataset privado con `movilidad_filtrado.parquet`, `movilidad_soluciones_filtrado.parquet`, `clasificar_vllm.py`, `ejemplos_movilidad.json` y `movilidad-referencia.json`.

**Estado (26/11/2026):** listo y sin correr. La versión de vLLM se fija en la parte B, después de verificar que corre en T4 con AWQ y salida guiada.

In [ ]:
# Versión a fijar en la parte B, cuando se verifique en T4.
!pip install -q vllm

In [ ]:
import glob, shutil, subprocess
from pathlib import Path

VERSION = "v2"  # igual que scripts/clasificacion/version.py
ENTRADA = Path(glob.glob("/kaggle/input/**/clasificar_vllm.py", recursive=True)[0]).parent
TRABAJO = Path("/kaggle/working")
SALIDA = TRABAJO / "resultados" / f"{VERSION}-movilidad"
for nombre in ["clasificar_vllm.py", "ejemplos_movilidad.json"]:
    shutil.copy(ENTRADA / nombre, TRABAJO / nombre)
ENTRADAS = [str(ENTRADA / "movilidad_filtrado.parquet"), str(ENTRADA / "movilidad_soluciones_filtrado.parquet")]
REFERENCIA = str(ENTRADA / "movilidad-referencia.json")
print(ENTRADA, SALIDA)

## 1. Gate: solo las 300 notas de la referencia

Mide la velocidad y produce respuestas para evaluar localmente contra la referencia. Usa otra carpeta de salida para no mezclar checkpoints con la corrida completa.

In [ ]:
!python clasificar_vllm.py --entradas {" ".join(ENTRADAS)} --salida {SALIDA}/gate --ids {REFERENCIA}

## 2. Corrida completa

Si se corta, volver a ejecutar esta celda: retoma desde los checkpoints de `SALIDA/checkpoints`.

In [ ]:
!python clasificar_vllm.py --entradas {" ".join(ENTRADAS)} --salida {SALIDA}

In [ ]:
import pandas as pd
df = pd.read_parquet(SALIDA / "movilidad_clasificado.parquet")
print(df["motivo"].value_counts(), df["tipo"].value_counts(), sep="

")
df[df["relevante"] == True].head(10)